# Aula 3A — Engenharia de Contexto com agentes reais

Na Aula 2, agentes especializados passaram a decidir como investigar um incidente e quais
ferramentas utilizar. Aqui, essa agência é preservada.

A pergunta é:

> Como controlar o contexto sem retirar do agente a decisão sobre o que investigar?

O agente escolhe as tools. O sistema limita acesso, tamanho, orçamento e rastreabilidade.

## Arquitetura

```text
incidente
  ↓
agente investigador
  ├── lista arquivos
  ├── busca no target_project
  └── lê arquivos selecionados
  ↓
agente revisor
  ↓
JSON estruturado
```

In [1]:
from pathlib import Path
import json, os, sys
from dotenv import load_dotenv

In [2]:
def find_course_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "target_project" / "mini_orders_pipeline").is_dir():
            return candidate
    raise FileNotFoundError("Nao foi encontrada a pasta target_project/mini_orders_pipeline/.")

COURSE_ROOT = find_course_root()
TARGET_PROJECT = COURSE_ROOT / "target_project/mini_orders_pipeline"
AULA_3_DIR = COURSE_ROOT / "Aula 3"

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

dotenv_candidates = [AULA_3_DIR / ".env", COURSE_ROOT / ".env"]
for env_path in dotenv_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        break

## Catálogo do projeto

A indexação é determinística porque sua função é organizar informação. A decisão de buscar,
selecionar e interpretar evidências permanece com o agente.

In [3]:
from shared.aula_3.project_access import build_project_catalog

catalog = build_project_catalog(TARGET_PROJECT)
print("Arquivos indexados:", len(catalog["documents"]))
[document["relative_path"] for document in catalog["documents"][:10]]

Arquivos indexados: 13


['.pytest_cache\\README.md',
 'data_samples\\pedidos_com_campo_ausente.json',
 'data_samples\\pedidos_com_data_regional.json',
 'data_samples\\pedidos_com_tipo_invalido.json',
 'data_samples\\pedidos_validos.json',
 'pyproject.toml',
 'README.md',
 'src\\mini_orders_pipeline\\ingestao_pedidos.py',
 'src\\mini_orders_pipeline\\notificacao_falhas.py',
 'src\\mini_orders_pipeline\\transformacao_pedidos.py']

## Ferramentas controladas

As tools são somente leitura, rejeitam caminhos fora de `target_project` e registram chamadas.
O agente não recebe acesso livre ao sistema de arquivos.

In [4]:
from shared.aula_3.crewai_components import build_project_tools

tool_bundle = build_project_tools(TARGET_PROJECT, catalog, event_log=[])
project_tools = tool_bundle["tools"]
tool_events = tool_bundle["events"]

[tool.name for tool in project_tools]

['list_target_project_files',
 'search_target_project',
 'read_target_project_file']

## Incidente e política de contexto

A política restringe o espaço de ação, mas não escolhe arquivos pelo agente.

In [5]:
incident = {
    "incident_id": "INC-AULA3-001",
    "title": "Falha em pipeline do target project",
    "description": (
        "Investigue logs, configuração, código e runbooks disponíveis e identifique "
        "a hipótese mais bem sustentada."
    ),
}

context_policy = '''
Use recuperação progressiva:
1. comece por busca textual;
2. leia apenas os arquivos mais promissores;
3. prefira evidência diretamente ligada ao incidente;
4. pare quando houver sustentação suficiente;
5. declare insuficiência quando necessário.
'''

## Construção e execução da crew



In [6]:
from shared.aula_3.crewai_components import (
    build_investigation_crew,
    build_llm,
    parse_crew_json,
)

MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

llm = build_llm(MODEL_NAME, temperature=0)
crew = build_investigation_crew(
    incident=incident,
    tools=project_tools,
    llm=llm,
    context_policy=context_policy,
    reviewer=True,
    verbose=True,
)


crew_result = crew.kickoff()
analysis = parse_crew_json(crew_result)
analysis

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 06db08d3-78e1-4917-ba06-25569a31c9c6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Investigue o incidente abaixo.                                                                                 │
│                                                                                                                 │
│  INCIDENTE:                                                                                                     │
│  {                                                                                                              │
│    "incident_id": "INC-AULA3-001",                                                                              │
│    "title": "Falha em pipeline do target project",                                                              │
│    "description": "Investigue logs, configuração, código e runbooks disponíveis e identifique a hipótese mais   │
│  bem sustentada."                                                                                               │
│  }                                                                                                              │
│                                                                                                                 │
│  POLÍTICA DE CONTEXTO:                                                                                          │
│                                                                                                                 │
│  Use recuperação progressiva:                                                                                   │
│  1. comece por busca textual;                                                                                   │
│  2. leia apenas os arquivos mais promissores;                                                                   │
│  3. prefira evidência diretamente ligada ao incidente;                                                          │
│  4. pare quando houver sustentação suficiente;                                                                  │
│  5. declare insuficiência quando necessário.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  Você decide quais ferramentas utilizar. Não receba o conteúdo integral do projeto                              │
│  antecipadamente. Busque e leia somente os arquivos necessários.                                                │
│                                                                                                                 │
│  Requisitos:                                                                                                    │
│  - cite identificadores no formato project:caminho/arquivo;                                                     │
│  - diferencie fatos, hipóteses e lacunas;                                                                       │
│  - não invente evidências;                                                                                      │
│  - não recomende ações destrutivas ou de produção sem revisão humana.                                           │
│                                                                                                                 │
│  ID: 45e9073b-73e8-4e73-bc08-3f4b220f2863                                                                       │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador técnico de incidentes                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Investigue o incidente abaixo.                                                                                 │
│                                                                                                                 │
│  INCIDENTE:                                                                                                     │
│  {                                                                                                              │
│    "incident_id": "INC-AULA3-001",                                                                              │
│    "title": "Falha em pipeline do target project",                                                              │
│    "description": "Investigue logs, configuração, código e runbooks disponíveis e identifique a hipótese mais   │
│  bem sustentada."                                                                                               │
│  }                                                                                                              │
│                                                                                                                 │
│  POLÍTICA DE CONTEXTO:                                                                                          │
│                                                                                                                 │
│  Use recuperação progressiva:                                                                                   │
│  1. comece por busca textual;                                                                                   │
│  2. leia apenas os arquivos mais promissores;                                                                   │
│  3. prefira evidência diretamente ligada ao incidente;                                                          │
│  4. pare quando houver sustentação suficiente;                                                                  │
│  5. declare insuficiência quando necessário.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  Você decide quais ferramentas utilizar. Não receba o conteúdo integral do projeto                              │
│  antecipadamente. Busque e leia somente os arquivos necessários.                                                │
│                                                                                                                 │
│  Requisitos:                                                                                                    │
│  - cite identificadores no formato project:caminho/arquivo;                                                     │
│  - diferencie fatos, hipóteses e lacunas;                                                                       │
│  - não invente evidências;                                                                                      │
│  - não recomende ações destrutivas ou de produção sem revisão humana.                                           │
│                                                                                                                 │
│                                                        

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: list_target_project_files                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Revisor de confiabilidade                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "incident_id": "INC-AULA3-001",                                                                              │
│    "classification": "Falha em pipeline do target project",                                                     │
│    "summary": "O incidente refere-se a uma falha em um pipeline simplificado de pedidos de e-commerce. O        │
│  projeto é estruturado em módulos que processam dados de pedidos, e a investigação inicial se concentrou em     │
│  entender o fluxo e as notificações de falhas.",                                                                │
│    "hypotheses": [                                                                                              │
│      "A falha pode estar relacionada a um erro em um dos módulos do pipeline, como ingestao_pedidos.py,         │
│  validacao_schema.py, transformacao_pedidos.py ou notificacao_falhas.py.",                                      │
│      "A notificação de falhas pode não estar sendo gerada corretamente, o que impede a identificação de         │
│  problemas no fluxo."                                                                                           │
│    ],                                                                                                           │
│    "evidence_ids": [                                                                                            │
│      "project:README.md",                                                                                       │
│      "project:src\\mini_orders_pipeline\\notificacao_falhas.py"                                                 │
│    ],                                                                                                           │
│    "recommended_actions": [                                                                                     │
│      "Realizar uma revisão humana para validar as hipóteses e propor correções antes de qualquer alteração no   │
│  código ou na configuração do pipeline."                                                                        │
│    ],                                                                                                           │
│    "confidence": 0.5,                                                                                           │
│    "requires_human_review": true                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Revise a investigação anterior.                                                                                │
│                                                                                                                 │
│  Verifique:                                                                                                     │
│  1. se as evidências citadas foram realmente consultadas;                                                       │
│  2. se as hipóteses são proporcionais às evidências;                                                            │
│  3. se a confiança é coerente;                                                                                  │
│  4. se riscos ou falta de informação exigem revisão humana.                                                     │
│                                                                                                                 │
│  Responda SOMENTE com JSON válido, sem markdown, obedecendo exatamente:                                         │
│  {                                                                                                              │
│    "incident_id": "string",                                                                                     │
│    "classification": "string",                                                                                  │
│    "summary": "string",                                                                                         │
│    "hypotheses": ["string"],                                                                                    │
│    "evidence_ids": ["project:caminho"],                                                                         │
│    "recommended_actions": ["string"],                                                                           │
│    "confidence": 0.0,                                                                                           │
│    "requires_human_review": true                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
│  Agent: Revisor de confiabilidade                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 06db08d3-78e1-4917-ba06-25569a31c9c6                                                                       │
│  Final Output: {                                                                                                │
│    "incident_id": "INC-AULA3-001",                                                                              │
│    "classification": "Falha em pipeline do target project",                                                     │
│    "summary": "O incidente refere-se a uma falha em um pipeline simplificado de pedidos de e-commerce. O        │
│  projeto é estruturado em módulos que processam dados de pedidos, e a investigação inicial se concentrou em     │
│  entender o fluxo e as notificações de falhas.",                                                                │
│    "hypotheses": [                                                                                              │
│      "A falha pode estar relacionada a um erro em um dos módulos do pipeline, como ingestao_pedidos.py,         │
│  validacao_schema.py, transformacao_pedidos.py ou notificacao_falhas.py.",                                      │
│      "A notificação de falhas pode não estar sendo gerada corretamente, o que impede a identificação de         │
│  problemas no fluxo."                                                                                           │
│    ],                                                                                                           │
│    "evidence_ids": [                                                                                            │
│      "project:README.md",                                                                                       │
│      "project:src\\mini_orders_pipeline\\notificacao_falhas.py"                                                 │
│    ],                                                                                                           │
│    "recommended_actions": [                                                                                     │
│      "Realizar uma revisão humana para validar as hipóteses e propor correções antes de qualquer alteração no   │
│  código ou na configuração do pipeline."                                                                        │
│    ],                                                                                                           │
│    "confidence": 0.5,                                                                                           │
│    "requires_human_review": true                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'incident_id': 'INC-AULA3-001',
 'classification': 'Falha em pipeline do target project',
 'summary': 'O incidente refere-se a uma falha em um pipeline simplificado de pedidos de e-commerce. O projeto é estruturado em módulos que processam dados de pedidos, e a investigação inicial se concentrou em entender o fluxo e as notificações de falhas.',
 'hypotheses': ['A falha pode estar relacionada a um erro em um dos módulos do pipeline, como ingestao_pedidos.py, validacao_schema.py, transformacao_pedidos.py ou notificacao_falhas.py.',
  'A notificação de falhas pode não estar sendo gerada corretamente, o que impede a identificação de problemas no fluxo.'],
 'evidence_ids': ['project:README.md',
  'project:src\\mini_orders_pipeline\\notificacao_falhas.py'],
 'recommended_actions': ['Realizar uma revisão humana para validar as hipóteses e propor correções antes de qualquer alteração no código ou na configuração do pipeline.'],
 'confidence': 0.5,
 'requires_human_review': True}

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Ações observáveis e custo de contexto

O log registra tools e entradas. Isso permite comparar políticas de contexto sem expor
raciocínio privado do modelo.

In [7]:
import pandas as pd
from shared.aula_3.token_budget import estimate_tokens

display(pd.DataFrame(tool_events))

estimated_tool_tokens = sum(
    estimate_tokens(event["tool_output_preview"])
    for event in tool_events
)
print("Tokens aproximados nos previews:", estimated_tool_tokens)
print("Uso reportado:", getattr(crew_result, "token_usage", None))

,tool_name,tool_input,tool_output_preview
0,list_target_project_files,{},"['.pytest_cache\\README.md', 'data_samples\\pe..."
1,search_target_project,"{'query': 'falha', 'top_k': 5}",[{'evidence_id': 'project:src\\mini_orders_pip...
2,search_target_project,"{'query': 'pipeline', 'top_k': 5}","[{'evidence_id': 'project:README.md', 'relativ..."
3,read_target_project_file,{'relative_path': 'src\mini_orders_pipeline\no...,"{'ok': True, 'evidence_id': 'project:src\\mini..."
4,read_target_project_file,{'relative_path': 'README.md'},"{'ok': True, 'evidence_id': 'project:README.md..."
5,search_target_project,"{'query': 'README.md', 'top_k': 1}",[{'evidence_id': 'project:.pytest_cache\\READM...
6,search_target_project,"{'query': 'notificacao_falhas.py', 'top_k': 1}",[{'evidence_id': 'project:src\\mini_orders_pip...


Tokens aproximados nos previews: 1241
Uso reportado: total_tokens=14270 prompt_tokens=12448 cached_prompt_tokens=2304 completion_tokens=1822 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=12
